In [ ]:
# Load every *_history.json and plot train vs val IoU
history_files = sorted(LOGS_DIR.glob('*_history.json'))
n = len(history_files)
print(f'Found {n} training histories')

if n == 0:
    print('No training histories yet — run training jobs on Isaac first.')
else:
    # Grid of subplots
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), squeeze=False)

    for idx, hf in enumerate(history_files):
        with open(hf) as f:
            hist = json.load(f)
        if not hist or 'train_iou' not in hist[0]:
            continue
        epochs    = [h['epoch'] for h in hist]
        train_iou = [h['train_iou'] for h in hist]
        val_iou   = [h['val_iou']   for h in hist]

        ax = axes[idx // cols][idx % cols]
        ax.plot(epochs, train_iou, label='Train IoU', color='steelblue', linewidth=1.5)
        ax.plot(epochs, val_iou,   label='Val IoU',   color='coral',     linewidth=1.5)
        ax.fill_between(epochs, train_iou, val_iou,
                        where=[t>v for t,v in zip(train_iou, val_iou)],
                        color='red', alpha=0.15, label='Train > Val (overfit signal)')
        ax.set_title(hf.stem.replace('_history',''), fontsize=10)
        ax.set_xlabel('Epoch'); ax.set_ylabel('IoU')
        ax.set_ylim(0, 1.0)
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(alpha=0.3)

    # Hide unused subplots
    for j in range(n, rows*cols):
        axes[j // cols][j % cols].set_visible(False)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'overfitting_diagnostic.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print final-epoch gap for each
    print('\nFinal-epoch train/val IoU gap (positive = overfitting risk):')
    for hf in history_files:
        with open(hf) as f:
            hist = json.load(f)
        if hist and 'iou_gap' in hist[-1]:
            print(f"  {hf.stem.replace('_history',''):<35} gap = {hist[-1]['iou_gap']:+.3f}")

## 2b. Overfitting Diagnostic — Train IoU vs Val IoU

Plot the train_iou vs val_iou per epoch for each model. A growing gap
between train and val IoU is the canonical overfitting signal. We do
NOT early-stop based on this; instead, we always select the best model
by val IoU, and use this plot to defend that the gap is healthy.

# Notebook 05 — Ablation Study & Uncertainty Analysis

**Runs locally after Isaac training completes.** Pulls results from `results/logs/` and produces the paper's analysis figures.

**Sections:**
1. Load all experiment results
2. Main results table (Otsu / FCN / Fusion / TriModal)
3. Ablation study — modality contribution
4. Cross-region generalization (Bolivia)
5. MC Dropout uncertainty — reliability diagram & ECE
6. Uncertainty map gallery

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))

LOGS_DIR = Path('../results/logs')
FIG_DIR  = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Run the aggregator to ensure all_results.csv is up to date
import subprocess
subprocess.run(['python', '../scripts/compile_results.py'], check=True)

df = pd.read_csv(LOGS_DIR / 'all_results.csv')
print(f'{len(df)} experiments loaded')
df.head()

## 2. Main Results Table

Compare Otsu → FCN → Fusion → TriModal on the test set.
This is the headline result.

In [ ]:
# Filter to the core models on test split
core = df[(df['split'] == 'test') &
          df['model'].str.contains('otsu|fcn|fusion_unet|trimodal_unet', regex=True, na=False) &
          ~df['model'].str.contains('ablation', na=False)]

display(core[['model', 'iou', 'dice', 'precision', 'recall']].round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = core['model'].tolist()
ious  = core['iou'].tolist()
colors = ['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c'][:len(names)]
bars = ax.bar(names, ious, color=colors, edgecolor='white')
for b, v in zip(bars, ious):
    ax.text(b.get_x()+b.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', fontweight='bold')
ax.set_ylabel('Test IoU'); ax.set_ylim(0, 1.0)
ax.set_title('Model Progression on Sen1Floods11 Test Set')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'main_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ablation Study — Does Each Modality Contribute?

7 ablation variants, all with identical EarlyFusionUNet architecture.
The only variable is which modalities are fed in. This isolates the
contribution of each modality, independent of architecture choices.

In [ ]:
abl = df[(df['split'] == 'test') &
         df['model'].str.startswith('ablation_', na=False)].copy()

# Extract modality string from model name: ablation_s1_s2_dem -> s1_s2_dem
abl['modset'] = abl['model'].str.replace('ablation_', '', regex=False)

# Ordering — single modalities, then pairs, then triple
order = ['s1', 's2', 'dem', 's1_s2', 's1_dem', 's2_dem', 's1_s2_dem']
abl['_sort'] = abl['modset'].apply(lambda x: order.index(x) if x in order else 99)
abl = abl.sort_values('_sort').drop(columns='_sort')

display(abl[['modset', 'iou', 'dice', 'precision', 'recall']].round(4))

# Stacked bar: show IoU gain from adding each modality
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(abl)))
bars = ax.bar(abl['modset'], abl['iou'], color=colors, edgecolor='white')
for b, v in zip(bars, abl['iou']):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}',
            ha='center', fontsize=9)
ax.set_xlabel('Modality subset'); ax.set_ylabel('Test IoU')
ax.set_title('Ablation Study — Per-Modality Contribution')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ablation_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Marginal gain analysis: how much does each modality add on top of others?
iou_map = dict(zip(abl['modset'], abl['iou']))

if all(k in iou_map for k in ['s1', 's1_s2', 's1_dem', 's1_s2_dem']):
    print('Marginal gains from adding modalities to S1 baseline:')
    print(f"  + S2  alone: {iou_map['s1_s2']  - iou_map['s1']:+.4f}")
    print(f"  + DEM alone: {iou_map['s1_dem'] - iou_map['s1']:+.4f}")
    print(f"  + S2 + DEM : {iou_map['s1_s2_dem'] - iou_map['s1']:+.4f}")
    print()
    print('Is DEM additive to S2?')
    print(f"  S1+S2      = {iou_map['s1_s2']:.4f}")
    print(f"  S1+S2+DEM  = {iou_map['s1_s2_dem']:.4f}")
    print(f"  Δ from adding DEM: {iou_map['s1_s2_dem'] - iou_map['s1_s2']:+.4f}")

## 4. Cross-Region Generalization — Bolivia

Bolivia is held out from Sen1Floods11's train/val/test splits.
Testing on Bolivia shows whether the model learned transferable
flood physics or just memorized the 10 training flood events.

In [ ]:
bol = df[df['split'] == 'bolivia']
test = df[df['split'] == 'test']

if len(bol) == 0:
    print('No Bolivia results yet — run slurm/eval_bolivia.sbatch on Isaac first.')
else:
    # Join test and bolivia on model name to compute gap
    merged = test[['model', 'iou']].merge(
        bol[['model', 'iou']], on='model', suffixes=('_test', '_bolivia')
    )
    merged['drop'] = merged['iou_test'] - merged['iou_bolivia']
    display(merged.round(4))

    # Side-by-side bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(merged))
    w = 0.4
    ax.bar(x - w/2, merged['iou_test'],    w, label='Test IoU',    color='#5bc0de')
    ax.bar(x + w/2, merged['iou_bolivia'], w, label='Bolivia IoU', color='#d9534f')
    ax.set_xticks(x); ax.set_xticklabels(merged['model'], rotation=15)
    ax.set_ylabel('IoU'); ax.set_ylim(0, 1.0)
    ax.set_title('Cross-Region Generalization: Test vs Bolivia')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'cross_region.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. MC Dropout Uncertainty — Reliability & ECE

Low ECE = model's confidence is accurate. This is the evidence that
the model can be trusted for decision support.

In [ ]:
from src.utils.uncertainty import reliability_diagram

unc_files = list(LOGS_DIR.glob('*_uncertainty.json'))
if not unc_files:
    print('No uncertainty results yet — run slurm/mc_uncertainty.sbatch on Isaac first.')
else:
    for f in unc_files:
        with open(f) as fp:
            u = json.load(fp)
        print(f"\n=== {f.stem} ===")
        print(f"Model: {u['model']}  |  Split: {u['split']}")
        print(f"ECE:   {u['ece']:.4f}  (lower = better calibrated)")
        print(f"Mean predictive uncertainty: {u['mean_uncertainty']:.4f}")

        fig = reliability_diagram(u['bins'], u['ece'],
                                  save_path=FIG_DIR / f"reliability_{u['model']}_{u['split']}.png")
        plt.show()

## 6. Uncertainty Map Gallery

Qualitative confirmation that high uncertainty coincides with visually
ambiguous regions (flood boundaries, cloud shadows, speckle). These
PNGs are generated by `scripts/mc_uncertainty.py`.

In [ ]:
from IPython.display import Image, display

unc_fig_dirs = list(FIG_DIR.glob('uncertainty_*'))
for d in unc_fig_dirs:
    print(f'\n{d.name}')
    for img in sorted(d.glob('uncertainty_map_*.png'))[:3]:
        display(Image(filename=str(img)))